# Accessing MMS Data
author: Louis Richard\
pyrfu reads MMS data from three resources:

* **local**: CDF files on a disk, in the MMS Science Data Center (SDC)
  directory tree (`mms1/fgm/brst/l2/2015/10/30/...`);
* **sdc**: the MMS SDC at LASP, read over the internet (public data, or
  SITL data with credentials);
* **aws**: the public copy of the MMS archive on NASA HelioCloud (Amazon S3),
  read without AWS credentials.

The same functions (`mms.get_data`, `mms.db_get_ts`, ...) work with all three.
This example lists, loads and downloads public data, and then reads the
downloaded files locally.

In [1]:
import os

import numpy as np

from pyrfu import mms

## Time interval and spacecraft
A short interval of burst mode data on 2015-10-30.

In [2]:
tint = ["2015-10-30T05:15:40.000", "2015-10-30T05:15:50.000"]
mms_id = 3

Burst mode data only exist in the burst segments selected for download. The
segments overlapping an interval are listed by the SDC.

In [3]:
mms.load_brst_segments(["2015-10-30T05:00:00", "2015-10-30T05:30:00"])

[['2015-10-30T05:14:44.000000000', '2015-10-30T05:17:44.000000000']]

## Default resource
`mms.db_init` saves the default resource and the local data path in the
configuration file of the user (`pyrfu.mms.MMS_CFG_PATH`), and the SDC
credentials (SITL access only) in the system keyring:

```python
mms.db_init(default="aws")  # read from AWS by default
mms.db_init(default="local", local="/path/to/mms/data")  # or from a disk
```

This example doesn't change the configuration: it passes the resource
(`source`) and the local path (`data_path`) explicitly to each function.

## List the files
The variables are named with keys made of the quantity, the coordinate
system, the instrument, the data rate and the level (see `mms.get_data`).
The files containing a variable are listed from each resource with the
tokenized key.

In [4]:
var = mms.tokenize("b_gse_fgm_brst_l2")
var

{'param': 'b',
 'cs': 'gse',
 'inst': 'fgm',
 'tmmode': 'brst',
 'lev': 'l2',
 'dtype': '',
 'cdf_name': 'fgm_b_gse_brst_l2'}

### SDC (public)

In [5]:
files_sdc = mms.list_files_sdc(tint, mms_id, var)
[file["file_name"] for file in files_sdc]

['mms3_fgm_brst_l2_20151030051444_v4.18.0.cdf']

### AWS (NASA HelioCloud)

In [6]:
files_aws = mms.list_files_aws(tint, mms_id, var)
[file["full_name"] for file in files_aws]

['spdf/cdaweb/data/mms/mms3/fgm/brst/l2/2015/10/mms3_fgm_brst_l2_20151030051444_v4.18.0.cdf']

## Load data without saving the files
With `source="sdc"` or `source="aws"`, the files are read into memory: nothing
is written to the disk. Both resources give the same data.

In [7]:
b_sdc = mms.get_data("b_gse_fgm_brst_l2", tint, mms_id, source="sdc")
b_aws = mms.get_data("b_gse_fgm_brst_l2", tint, mms_id, source="aws")

print(b_aws.shape, np.array_equal(b_sdc.data, b_aws.data))

[06-Oct-26 17:58:57] INFO: Loading mms3_fgm_b_gse_brst_l2...


[06-Oct-26 17:59:14] INFO: Loading mms3_fgm_b_gse_brst_l2...


(1280, 3) True


Particle distributions are loaded the same way, here the ion skymap.

In [8]:
vdf_i = mms.get_data("pdi_fpi_brst_l2", tint, mms_id, source="aws")
vdf_i

[06-Oct-26 17:59:25] INFO: Loading mms3_dis_dist_brst...


<xarray.Dataset> Size: 4MB
Dimensions:  (time: 66, idx0: 32, idx1: 32, idx2: 16)
Coordinates:
  * time     (time) datetime64[ns] 528B 2015-10-30T05:15:40.114711 ... 2015-1...
  * idx0     (idx0) int64 256B 0 1 2 3 4 5 6 7 8 ... 23 24 25 26 27 28 29 30 31
  * idx1     (idx1) int64 256B 0 1 2 3 4 5 6 7 8 ... 23 24 25 26 27 28 29 30 31
  * idx2     (idx2) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Data variables:
    data     (time, idx0, idx1, idx2) float32 4MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    phi      (time, idx1) float32 8kB 6.25 17.5 28.75 40.0 ... 333.9 345.1 356.4
    theta    (idx2) float32 64B 5.625 16.88 28.12 39.38 ... 151.9 163.1 174.4
    energy   (time, idx0) float32 8kB 12.06 15.49 19.89 ... 1.944e+04 2.497e+04
Attributes: (12/38)
    Acknowledgement:                  []
    Correction_table_name:            mms3_dis_f1ct_v22-v24-0223_0302_p0.txt ...
    Correction_table_rev:             7825
    Correction_table_scaling_factor:  1.00000
    Data_type:                        brst_l2_dis-dist
    Data_version:                     3.3.0
    ...                               ...
    delta_energy_plus:                [[1.71000e+00 1.71000e+00 2.69000e+00 ....
    energy0:                          [1.064000e+01 1.366000e+01 1.755000e+01...
    energy1:                          [1.206000e+01 1.549000e+01 1.989000e+01...
    esteptable:                       [1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 ...
    species:                          ions
    tmmode:                           brst

## Download the files
`mms.download_data` saves the files from the SDC (public access by default)
into a local directory, in the SDC directory tree. Here, a new directory
`mms_data` in the home directory.

In [9]:
data_path = os.path.join(os.path.expanduser("~"), "mms_data")
os.makedirs(data_path, exist_ok=True)

for var_str in ["b_gse_fgm_brst_l2", "ni_fpi_brst_l2"]:
    mms.download_data(var_str, tint, mms_id, data_path=data_path)

[06-Oct-26 17:59:33] INFO: Downloading mms3_fgm_brst_l2_20151030051444_v4.18.0.cdf from https://lasp.colorado.edu/mms/sdc/public/files/api/v1/download/science?file=mms3_fgm_brst_l2_20151030051444_v4.18.0.cdf...


  0%|                           | 0.00/1.56M [00:00<?, ?B/s]

 67%|████████████▊      | 1.05M/1.56M [00:06<00:03, 161kB/s]

100%|███████████████████| 1.56M/1.56M [00:11<00:00, 134kB/s]

100%|███████████████████| 1.56M/1.56M [00:11<00:00, 138kB/s]

[06-Oct-26 17:59:45] INFO: Downloading mms3_fpi_brst_l2_dis-moms_20151030051444_v3.3.0.cdf from https://lasp.colorado.edu/mms/sdc/public/files/api/v1/download/science?file=mms3_fpi_brst_l2_dis-moms_20151030051444_v3.3.0.cdf...


  0%|                           | 0.00/1.71M [00:00<?, ?B/s]

 61%|███████████▋       | 1.05M/1.71M [00:05<00:03, 184kB/s]

100%|███████████████████| 1.71M/1.71M [00:08<00:00, 213kB/s]

100%|███████████████████| 1.71M/1.71M [00:08<00:00, 207kB/s]

Ancillary files (definitive attitude and ephemeris, ...) are downloaded with
`mms.download_ancillary`.

In [10]:
mms.download_ancillary("defeph", tint, mms_id, data_path=data_path)

[06-Oct-26 17:59:55] INFO: Downloading MMS3_DEFEPH_2015303_2015304.V01 from https://lasp.colorado.edu/mms/sdc/public/files/api/v1/download/ancillary?file=MMS3_DEFEPH_2015303_2015304.V01...


  0%|                            | 0.00/531k [00:00<?, ?B/s]

100%|█████████████████████| 531k/531k [00:03<00:00, 164kB/s]

100%|█████████████████████| 531k/531k [00:03<00:00, 164kB/s]


[06-Oct-26 17:59:59] INFO: Downloading MMS3_DEFEPH_2015302_2015303.V01 from https://lasp.colorado.edu/mms/sdc/public/files/api/v1/download/ancillary?file=MMS3_DEFEPH_2015302_2015303.V01...


  0%|                            | 0.00/528k [00:00<?, ?B/s]

100%|█████████████████████| 528k/528k [00:02<00:00, 237kB/s]

100%|█████████████████████| 528k/528k [00:02<00:00, 237kB/s]

The downloaded files:

In [11]:
for root, _, files in sorted(os.walk(data_path)):
    for file in sorted(files):
        print(os.path.relpath(os.path.join(root, file), data_path))

ancillary/mms3/defeph/MMS3_DEFEPH_2015302_2015303.V01
ancillary/mms3/defeph/MMS3_DEFEPH_2015303_2015304.V01
mms3/fgm/brst/l2/2015/10/30/mms3_fgm_brst_l2_20151030051444_v4.18.0.cdf
mms3/fpi/brst/l2/dis-moms/2015/10/30/mms3_fpi_brst_l2_dis-moms_20151030051444_v3.3.0.cdf


## Load the local files
With `source="local"`, the files are read from `data_path` (or from the
local path of the configuration if it is not given).

In [12]:
b_local = mms.get_data(
    "b_gse_fgm_brst_l2", tint, mms_id, data_path=data_path, source="local"
)
n_i = mms.get_data("ni_fpi_brst_l2", tint, mms_id, data_path=data_path, source="local")

print(np.array_equal(b_local.data, b_aws.data))
n_i

[06-Oct-26 18:00:01] INFO: Loading mms3_fgm_b_gse_brst_l2...


[06-Oct-26 18:00:01] INFO: Loading mms3_dis_numberdensity_brst...


True


<xarray.DataArray (time: 66)> Size: 264B
array([ 2.2235613,  2.290725 ,  2.1579852,  2.317438 ,  2.3006346,
        2.3519325,  2.368275 ,  2.3398252,  2.3060496,  2.3273313,
        2.4409676,  2.401601 ,  2.4984367,  2.354684 ,  2.4053214,
        2.2436247,  2.0546083,  2.03952  ,  1.9515913,  1.9025486,
        1.9700876,  1.8699645,  1.9892935,  1.9991937,  2.1081152,
        2.220782 ,  2.1822999,  2.1964686,  2.433504 ,  2.6993158,
        2.8384435,  2.9472177,  3.226134 ,  3.9842284,  4.003387 ,
        4.2572546,  5.2805276,  6.0816994,  7.9601116,  9.197284 ,
       12.313461 , 15.133933 , 19.113491 , 22.54991  , 24.025536 ,
       25.192642 , 26.557158 , 27.086992 , 27.290533 , 26.308762 ,
       25.24935  , 24.58838  , 24.083744 , 23.404938 , 23.748816 ,
       24.452684 , 24.305983 , 24.514305 , 24.856607 , 27.412235 ,
       30.853418 , 32.08202  , 32.694347 , 33.04226  , 33.495537 ,
       34.618095 ], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 528B 2015-10-30T05:15:40.114711 ... 2015-1...
Attributes: (12/16)
    CATDESC:          MMS3 FPI/DIS ion number density during this burst
    DELTA_MINUS_VAR:  mms3_dis_numberdensity_err_brst
    DELTA_PLUS_VAR:   mms3_dis_numberdensity_err_brst
    DEPEND_0:         Epoch
    DISPLAY_TYPE:     time_series
    FIELDNAM:         MMS3 FPI/DIS number density
    ...               ...
    SCALETYP:         log
    SI_CONVERSION:    1e6>m^-3
    UNITS:            cm^-3
    VALIDMAX:         [100000.0]
    VALIDMIN:         [0.0]
    VAR_TYPE:         data

The local files are listed with `mms.list_files` (here relative to
`data_path`).

In [13]:
[
    os.path.relpath(file, data_path)
    for file in mms.list_files(tint, mms_id, var, data_path)
]

['mms3/fgm/brst/l2/2015/10/30/mms3_fgm_brst_l2_20151030051444_v4.18.0.cdf']

The ancillary files are read with `mms.load_ancillary`. The definitive
ephemeris has one sample every 30 s, so a few minutes are loaded here.

In [14]:
tint_long = ["2015-10-30T05:14:00.000", "2015-10-30T05:18:00.000"]
defeph = mms.load_ancillary(
    "defeph", tint_long, mms_id, verbose=False, data_path=data_path
)
defeph

<xarray.Dataset> Size: 512B
Dimensions:        (time: 8)
Coordinates:
  * time           (time) datetime64[ns] 64B 2015-10-30T05:14:24.227000 ... 2...
Data variables:
    epoch_mms_tai  (time) float64 64B 2.112e+04 2.112e+04 ... 2.112e+04
    x              (time) float64 64B -4.141e+04 -4.14e+04 ... -4.137e+04
    y              (time) float64 64B -4.686e+04 -4.69e+04 ... -4.714e+04
    z              (time) float64 64B -2.304e+04 -2.306e+04 ... -2.32e+04
    vx             (time) float64 64B 0.1817 0.1834 0.1851 ... 0.1917 0.1934
    vy             (time) float64 64B -1.372 -1.37 -1.368 ... -1.36 -1.358
    vz             (time) float64 64B -0.7636 -0.7626 -0.7617 ... -0.758 -0.7571